# Part 1


## Part 1 of N — Attention Mechanisms, Multi-Head Attention, Memory Optimization, and Positional Encodings


## 1. The Transformer's Core Computation: The Attention Mechanism


### 1.1 Why Attention? The Problem With Sequential Processing

Before transformers, sequence modeling was dominated by RNNs and LSTMs. Their fundamental flaw: **hidden state is a fixed-size bottleneck**. No matter how long the input sequence, you compress everything into a single vector $h_t$ before generating the next token. This creates an information bottleneck that degrades over long sequences (the "vanishing gradient over time" problem, distinct from depth-based vanishing gradients).

The key insight of attention: **instead of compressing the past into a single vector, allow every output position to directly attend to every input position**. This makes the dependency path between any two tokens $O(1)$ regardless of distance, compared to $O(n)$ for RNNs.

---

### 1.2 The QKV Framework: Formal Derivation

Every token $x_i$ in the input sequence is first embedded into a $d_{model}$-dimensional vector. Within each attention layer, three separate linear projections transform this embedding:

$$Q = XW^Q, \quad K = XW^K, \quad V = XW^V$$

where:
- $X \in \mathbb{R}^{n \times d_{model}}$ is the input sequence matrix
- $W^Q, W^K \in \mathbb{R}^{d_{model} \times d_k}$ are the query/key projection matrices
- $W^V \in \mathbb{R}^{d_{model} \times d_v}$ is the value projection matrix

The **query** $q_i$ represents *what token $i$ is looking for* — a vector in a learned "question" subspace.

The **key** $k_j$ represents *what token $j$ can offer* — a vector in a learned "advertisement" subspace.

The **value** $v_j$ is *the actual content* to be retrieved if there's a match.

The attention output for token $i$ is:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**Intuition on the dot product:** $q_i \cdot k_j = \|q_i\|\|k_j\|\cos\theta$. If $q$ and $k$ point in the same direction (high cosine similarity), you get a large positive score. Orthogonal vectors give ~0. The dot product is a similarity measure in high-dimensional space.

---

### 1.3 The Scaling Factor $\frac{1}{\sqrt{d_k}}$: Why It's Non-Negotiable

This is often treated as a minor implementation detail. It is not. It's foundational to training stability.

**The statistical argument:** Assume $q$ and $k$ are vectors of dimension $d_k$ with i.i.d. components drawn from $\mathcal{N}(0, 1)$. The dot product $q \cdot k = \sum_{i=1}^{d_k} q_i k_i$ has:

$$\mathbb{E}[q \cdot k] = 0, \quad \text{Var}[q \cdot k] = d_k$$

So the standard deviation grows as $\sqrt{d_k}$. For $d_k = 64$, std $= 8$. For $d_k = 512$, std $= 22.6$. Without scaling, the raw dot products can be in the range $[-30, +30]$ easily.

**Why this destroys softmax:** The softmax function is:

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

For large magnitude inputs, $e^{z_i}$ for the dominant element becomes astronomically large compared to others. The output distribution becomes a near-one-hot vector. Backpropagating through a near-one-hot softmax means the gradient $\frac{\partial \text{softmax}}{\partial z}$ is approximately zero everywhere — **the vanishing gradient problem caused by softmax saturation.**

Dividing by $\sqrt{d_k}$ normalizes the variance back to 1, keeping inputs to softmax in a regime where the distribution is smooth and gradients are non-zero.

---

### 1.4 Causal Masking (Decoder-Only)

In decoder-only models, attention must be **causal**: token $i$ cannot attend to token $j > i$. This is enforced by adding a mask before softmax:

$$\text{score}_{ij} = \begin{cases} q_i \cdot k_j / \sqrt{d_k} & \text{if } j \leq i \\ -\infty & \text{if } j > i \end{cases}$$

$-\infty$ becomes $0$ after softmax, so future tokens effectively don't exist. This is how **auto-regressive generation** is trained on full sequences in parallel — even though at inference time tokens are generated one at a time.

---

## 2. Multi-Head Attention (MHA)

### 2.1 Motivation: One Subspace Is Never Enough

A single attention head learns **one** linear projection of the relational structure. Language has multiple simultaneously relevant relationship types:
- **Syntactic:** subject-verb agreement, dependency parsing
- **Semantic:** word sense disambiguation (bank → river vs. bank → finance)
- **Coreference:** pronoun resolution across long spans
- **Positional/local:** n-gram patterns, idioms

Forcing all of this into a single $d_k$-dimensional space creates destructive interference. Multi-head attention is the solution.

### 2.2 Formal Specification

For $h$ heads, with $d_k = d_v = d_{model}/h$:

$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)W^O$$

where $W^O \in \mathbb{R}^{hd_v \times d_{model}}$ is the output projection.

**Dimensionality note:** Each head operates in a $d_k = 128$ subspace (for $d_{model}=4096$, $h=32$). The heads run **truly in parallel** on the GPU as batched matrix multiplications — there is no sequential dependency between heads.

**Parameter count for one MHA layer:**
- $W^Q, W^K, W^V$: $3 \times d_{model} \times d_{model}$
- $W^O$: $d_{model} \times d_{model}$
- Total: $4 d_{model}^2$ (same as four square matrices of the model dimension)

For $d_{model} = 4096$: $4 \times 4096^2 \approx 67M$ parameters per attention layer.

---

## 3. The KV Cache: The Production Bottleneck

### 3.1 Why It Exists

During auto-regressive decoding (inference), to generate token $t+1$, the model must compute attention over all tokens $1..t$. Naively recomputing all $K, V$ projections at every step costs $O(t)$ per step and $O(t^2)$ total. The **KV cache** stores all previously computed $K$ and $V$ vectors, making each step $O(1)$ in compute (amortized), but $O(t)$ in memory.

### 3.2 Memory Cost Analysis

For a model with:
- $L$ layers
- $h$ attention heads
- $d_k$ head dimension
- Sequence length $n$
- Batch size $B$
- Precision $p$ bytes (2 for FP16/BF16)

$$\text{KV Cache Size} = 2 \times L \times h \times d_k \times n \times B \times p$$

For Llama-2 70B ($L=80$, $h=64$, $d_k=128$, $n=4096$, $B=1$, $p=2$):

$$= 2 \times 80 \times 64 \times 128 \times 4096 \times 1 \times 2 \approx 10.7 \text{ GB}$$

For $B=10$ concurrent users: **107 GB** — just for the KV cache, before even storing model weights (~140 GB in FP16). This is why MHA is economically brutal at scale.

---

## 4. MQA → GQA: The Memory-Quality Tradeoff Spectrum

### 4.1 Multi-Query Attention (MQA)

**Shazeer, 2019.** Radical simplification: all $h$ query heads share **one** K head and **one** V head.

$$\text{head}_i = \text{Attention}(QW_i^Q, KW^K_{\text{shared}}, VW^V_{\text{shared}})$$

**Memory reduction:** KV cache scales by $1/h$ instead of $1$. For $h=64$: **64× smaller KV cache**.

**The cost:** Each query head asks a different question but looks at the same K/V representation. The model loses the ability to build diverse relational subspaces. Empirically: ~1-2% perplexity degradation on benchmarks, but **4-5× inference throughput improvement** due to reduced memory bandwidth pressure.

**Why bandwidth, not compute, is the bottleneck:** Modern GPUs (H100) have ~3.35 PFLOPS of BF16 compute but only ~3.35 TB/s of HBM bandwidth. For decoding, the arithmetic intensity (FLOPs per byte moved) is extremely low — you move gigabytes of KV cache to compute a handful of dot products. The GPU is bandwidth-bound, not compute-bound.

### 4.2 Grouped Query Attention (GQA)

**Ainslie et al., 2023.** Generalizes the spectrum: $g$ KV heads, with $h/g$ query heads sharing each KV head.

- $g = h$: Full MHA
- $g = 1$: MQA  
- $1 < g < h$: GQA (the sweet spot)

Llama-2 70B uses $h=64$ query heads, $g=8$ KV heads → 8 query heads share 1 KV pair. **KV cache is 8× smaller than MHA** while retaining ~98% of the quality.

Llama-3 and Mistral also use GQA as the default. It has become the de facto standard for production-scale models.

**Mathematical framing:** GQA is equivalent to MHA where the $W^K$ and $W^V$ matrices are block-diagonal with tied blocks. The expressivity reduction is bounded and empirically benign for most downstream tasks.

---

## 5. Flash Attention: Hardware-Aware Algorithm Design

### 5.1 The Problem: Memory Hierarchy Mismatch

Standard attention materializes the full $n \times n$ score matrix in HBM:

1. Compute $S = QK^T / \sqrt{d_k}$ → write $S$ to HBM ($O(n^2)$ memory)
2. Read $S$ from HBM, compute $P = \text{softmax}(S)$ → write $P$ to HBM
3. Read $P$ and $V$ from HBM, compute $O = PV$ → write $O$ to HBM

Every step is an HBM read/write. HBM bandwidth on an H100 is ~3.35 TB/s, but the round-trip latency for large matrices is the bottleneck. GPU SRAM (L1 cache + shared memory) is ~20 MB total but has ~19 TB/s effective bandwidth — nearly **6× faster**.

For $n=8192$, $d_k=128$: the $n \times n$ matrix is $8192^2 \times 2$ bytes $\approx$ **134 MB** — far too large for SRAM.

### 5.2 The Solution: Tiling + Online Softmax

Flash Attention (**Dao et al., 2022**) uses two key techniques:

**Tiling:** Break $Q$, $K$, $V$ into blocks of size $B_r \times d_k$ and $B_c \times d_k$ that fit in SRAM. Process block by block.

**Online softmax (the mathematical trick):** Softmax requires the full row sum $\sum_j e^{s_{ij}}$ to normalize. But you can maintain a running maximum $m$ and running sum $\ell$ and update them incrementally:

For block $t$ with max score $m_t^{\text{new}} = \max(m_t^{\text{old}}, \text{rowmax}(S_t))$:

$$\ell^{\text{new}} = e^{m^{\text{old}} - m^{\text{new}}} \ell^{\text{old}} + \sum_j e^{s_{ij} - m^{\text{new}}}$$

$$O^{\text{new}} = \frac{e^{m^{\text{old}} - m^{\text{new}}} \ell^{\text{old}} \cdot O^{\text{old}} + e^{S_t - m^{\text{new}}} V_t}{\ell^{\text{new}}}$$

This is numerically equivalent to the standard softmax but never needs the full row simultaneously.

**Kernel fusion:** The entire attention computation (QK multiply, mask, softmax, AV multiply) is fused into a single CUDA kernel, eliminating intermediate HBM round-trips.

**Result:**
- Memory: $O(n^2)$ → $O(n)$ (no full matrix materialized)
- Speed: **2-4× faster** than standard attention (even with ~same FLOPs), because GPU cores stop starving for data
- Enables context lengths of 100K+ tokens that were previously impossible

### 5.3 Flash Attention 2 & 3

**FA2** added better parallelism over the sequence length dimension and reduced non-matmul FLOPs. **FA3** (2024) targets H100's new features: asynchronous memory copies and FP8 compute, squeezing even more out of the hardware.

---

## 6. Sliding Window Attention

### 6.1 Architecture

Used in Mistral 7B. Each token attends only to the $W$ most recent tokens (window size $W = 4096$ in Mistral). The attention matrix is banded rather than lower-triangular.

**Complexity:** $O(n \cdot W)$ instead of $O(n^2)$.

### 6.2 The Receptive Field Argument

The concern: does the model "forget" the beginning of a 100K document?

With $L$ layers each with window $W$, the **effective receptive field** at layer $L$ is $L \times W$. For Mistral with $L=32$, $W=4096$: effective receptive field = $131,072$ tokens. The information from early tokens doesn't disappear — it propagates forward through intermediate representations at each layer.

This is analogous to dilated convolutions in CNNs: local operations at each layer, but exponentially growing receptive fields with depth.

**The limitation:** Unlike full attention where there's a *direct* gradient path between any two tokens, sliding window creates an *indirect* path through intermediate layers. Fine-grained token-to-token interaction across long distances requires multiple hops, which can degrade precision for certain retrieval tasks. This is the origin of the "lost in the middle" problem for very long contexts.

---

## 7. Positional Encodings

### 7.1 The Permutation Invariance Problem

Raw transformer attention computes $\text{softmax}(QK^T/\sqrt{d_k})V$. This operation is **permutation equivariant**: shuffle the input tokens, and the output is shuffled in the same way. The model has no intrinsic notion of "token 5 comes before token 6."

We must inject positional information explicitly.

### 7.2 Absolute Sinusoidal (Original Transformer)

$$PE_{(pos, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d_{model}}}\right), \quad PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

Different dimensions encode position at different frequencies (like a Fourier decomposition). The dot product between position embeddings at positions $p$ and $q$ depends only on $p - q$, giving some relative sensitivity.

**Failure mode:** At inference on sequences longer than the training length, the sinusoidal values extrapolate to positions the model was never trained on. Embeddings degrade. Performance collapses.

### 7.3 Learned Absolute Embeddings

Replace the formula with a learned lookup table $E \in \mathbb{R}^{n_{max} \times d_{model}}$. Works well within training distribution. **Same failure mode:** position indices $> n_{max}$ have no learned embedding.

### 7.4 RoPE (Rotary Positional Embedding)

**Su et al., 2021.** Used in Llama, PaLM, Falcon, and most modern models.

**Core idea:** Rather than adding a positional vector to the embedding, apply a **rotation** to the query and key vectors based on position. For a pair of dimensions $(2i, 2i+1)$ at position $m$:

$$\begin{pmatrix} q_{2i}^m \\ q_{2i+1}^m \end{pmatrix} = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix} \begin{pmatrix} q_{2i} \\ q_{2i+1} \end{pmatrix}$$

where $\theta_i = 10000^{-2i/d_k}$.

**The elegant property:** The dot product between rotated $q^m$ and $k^n$ depends only on the relative position $m - n$:

$$\langle R_m q, R_n k \rangle = \langle q, R_{n-m} k \rangle$$

The absolute positions cancel. The model learns purely relative relationships. It generalizes well beyond training length because the rotation angles are defined analytically (not via lookup table).

**RoPE scaling for long context:** To extend context beyond training length, you can scale the base $10000$ upward (e.g., to $500000$ in Llama-3), effectively compressing the rotation frequencies to allow longer relative positions to remain distinguishable.

### 7.5 ALiBi (Attention with Linear Biases)

**Press et al., 2021.** Does not modify embeddings at all. Instead, before softmax, subtracts a head-specific linear penalty proportional to token distance:

$$\text{score}_{ij} = q_i \cdot k_j / \sqrt{d_k} - m_h \cdot (i - j)$$

where $m_h$ is a head-specific slope (set geometrically across heads). Closer tokens pay less penalty; distant tokens pay more. This **hard-wires a local inductive bias** directly into the attention scores.

**Extrapolation advantage:** Because the bias is linear and algebraically simple, the model extrapolates to unseen sequence lengths smoothly. Used in BLOOM and MPT models.

**Tradeoff vs. RoPE:** ALiBi's linear decay may be too aggressive for tasks requiring long-range reasoning (e.g., retrieval over long documents). RoPE's decay is softer and more nuanced. RoPE has largely won the practical competition.

---

## 8. Normalization: Pre-Norm, Post-Norm, RMSNorm

### 8.1 Layer Norm

$$\text{LayerNorm}(x) = \frac{x - \mu}{\sigma + \epsilon} \cdot \gamma + \beta$$

where $\mu$ and $\sigma$ are computed per token across the feature dimension.

**Post-Norm (original transformer):** Applied after the residual: $x \leftarrow \text{LN}(x + \text{Sublayer}(x))$. The gradient must flow through the normalization, which rescales it. Deep networks (>12 layers) have unstable training with post-norm.

**Pre-Norm (modern default):** Applied before: $x \leftarrow x + \text{Sublayer}(\text{LN}(x))$. The residual stream $x$ passes through the network **unnormalized and uninterrupted**. Gradients flow back through the highway with zero impedance. Llama, GPT-NeoX, and virtually all modern models use pre-norm.

### 8.2 RMSNorm

$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \cdot \gamma, \quad \text{RMS}(x) = \sqrt{\frac{1}{d}\sum_{i=1}^d x_i^2}$$

Drops the mean-centering step from LayerNorm. **Justification:** Zhang & Sennrich (2019) showed that the re-centering step contributes negligible representational benefit but requires an extra pass over the data (a synchronization point on GPU). RMSNorm achieves **10–30% throughput improvement** with no measurable quality loss. Used in Llama and most recent architectures.

---

## End of Part 1

**Covered:** QKV mechanics, scaled dot-product attention, causal masking, MHA, KV cache analysis, MQA/GQA tradeoff, Flash Attention (tiling + online softmax), Sliding Window Attention, RoPE/ALiBi positional encodings, Pre-Norm, RMSNorm.

**Coming in Part 2:**
- Residual connections (the gradient highway in depth)
- Feed-Forward Networks (FFNs as memory banks) + SwiGLU
- Mixture of Experts (MoE), router collapse, load balancing loss
- Tokenization: BPE, Byte-Level BPE, structural bias
- Decoder-only architecture: the full case for why it won
- The training process: loss landscapes, saddle points, gradient pathologies

---


In [ ]:
# Part 2

# Part 2

## Part 2 — Feed-Forward Networks, MoE, Tokenization, Decoder Architecture & Training Dynamics



---

## 9. Feed-Forward Networks: The Underappreciated Majority

### 9.1 Structure and Parameter Dominance

Every transformer layer has two sub-components: the attention block and the Feed-Forward Network (FFN). Attention gets all the conceptual glory, but FFNs contain **roughly 2/3 of all parameters** in a dense transformer. For a model with $d_{model} = 4096$, the FFN typically expands to $d_{ff} = 4 \times d_{model} = 16384$:

$$\text{FFN}(x) = W_2 \cdot \sigma(W_1 x + b_1) + b_2$$

where $W_1 \in \mathbb{R}^{d_{ff} \times d_{model}}$, $W_2 \in \mathbb{R}^{d_{model} \times d_{ff}}$.

**Parameter count per FFN layer:**
$$2 \times d_{model} \times d_{ff} = 2 \times 4096 \times 16384 \approx 134M \text{ parameters}$$

Compare to the MHA layer at ~67M. For a 32-layer model: FFNs hold ~4.3B parameters vs. ~2.1B in attention.

### 9.2 FFNs as Key-Value Memory Banks

**Geva et al. (2021)** showed that FFN layers function as distributed key-value memory stores. The first weight matrix $W_1$ acts as **keys**: each row $w_i^{(1)}$ is a pattern detector. The nonlinearity + second matrix acts as **values**: if the pattern fires, $w_i^{(2)}$ (the corresponding column of $W_2$) is added to the residual stream.

Formally, if we denote the pre-activation as $h = W_1 x$, then:

$$\text{FFN}(x) = \sum_{i=1}^{d_{ff}} \sigma(h_i) \cdot w_i^{(2)}$$

Each neuron $i$ is a memory slot: $w_i^{(1)}$ is the retrieval key (matches certain input patterns), $\sigma(h_i)$ is the match strength, and $w_i^{(2)}$ is the value (the information to inject into the residual stream).

**Implication:** To know more facts, you scale $d_{ff}$. To reason better over existing facts, you scale depth. This is why scaling laws show parameter count and depth have different effects on different capability dimensions.

### 9.3 Activation Functions: From ReLU to SwiGLU

**ReLU:** $\sigma(x) = \max(0, x)$. Simple, gradient-friendly for positive activations, but causes **dead neurons** — units that get stuck at 0 if their pre-activation is always negative, permanently zeroing their gradient.

**GeLU (Gaussian Error Linear Unit):**
$$\text{GeLU}(x) = x \cdot \Phi(x) \approx 0.5x\left(1 + \tanh\!\left(\sqrt{2/\pi}(x + 0.044715x^3)\right)\right)$$

where $\Phi$ is the Gaussian CDF. GeLU is smooth, non-zero for slightly negative inputs, and empirically outperforms ReLU on language tasks (used in GPT-2, BERT).

**SwiGLU (Noam Shazeer, 2020):** The current standard in Llama, PaLM, Mistral.

$$\text{SwiGLU}(x) = \text{Swish}(W_1 x) \odot (W_3 x)$$

$$\text{Swish}(x) = x \cdot \sigma(x) = \frac{x}{1 + e^{-x}}$$

This is a **Gated Linear Unit**: the input is projected into two parallel streams. One passes through the Swish nonlinearity; the other remains linear. They are element-wise multiplied. The linear gate dynamically controls how much of the nonlinear activation passes through, acting as a **learned data-dependent filter**.

**Why it works:** The gating mechanism allows the network to suppress irrelevant activations precisely, producing sparser, more focused intermediate representations. Empirically yields ~1-3% perplexity improvement over GeLU at matched parameter count.

**Parameter cost:** SwiGLU requires three weight matrices ($W_1, W_2, W_3$) instead of two, but the hidden dimension is typically reduced to $\frac{2}{3} d_{ff}$ to keep total FLOPs constant. Llama uses $d_{ff} = \frac{8}{3} d_{model}$ rounded to a multiple of 256 for hardware alignment.

---

## 10. Mixture of Experts (MoE)

### 10.1 The Economic Problem With Dense Models

In a dense transformer, **every parameter is activated for every token**. For a 70B parameter model generating one token, all 70B parameters participate in the forward pass. This creates a hard coupling between model capacity (total parameters) and inference cost (active FLOPs per token). To know more, you pay more per query. Always.

MoE breaks this coupling.

### 10.2 Architecture

Replace each dense FFN with $E$ independent FFN "experts" plus a lightweight **router network**:

$$\text{MoE}(x) = \sum_{i \in \text{TopK}(G(x))} G_i(x) \cdot E_i(x)$$

where $G(x) = \text{softmax}(W_g x)$ is the router's output (probability distribution over experts), and $\text{TopK}$ selects the $k$ highest-probability experts (typically $k=2$).

**The Mixtral 8x7B example:**
- 8 experts per MoE layer, each is a 7B-parameter FFN
- Only 2 experts activated per token
- **Total parameters:** ~47B (all experts combined)
- **Active parameters per token:** ~13B (2 active experts + attention)
- **Inference cost:** comparable to a 13B dense model
- **Reasoning depth:** comparable to a 47B dense model

The economic efficiency is dramatic: you get the knowledge capacity of a large model at the inference cost of a small one.

### 10.3 Router Collapse and Load Balancing

**The pathology:** The router is differentiable and trained by gradient descent. If one expert happens to produce slightly better outputs early in training, the router learns to route more tokens to it. That expert trains more, becomes better, attracting even more tokens. This **positive feedback loop** causes all tokens to route to 1-2 experts, leaving the rest untrained. The model degrades to an expensive dense model with mostly dead capacity.

**The fix — Auxiliary Load Balancing Loss:**

Define $f_i$ as the fraction of tokens routed to expert $i$, and $P_i$ as the average router probability for expert $i$. Add:

$$\mathcal{L}_{\text{aux}} = \alpha \cdot E \cdot \sum_{i=1}^{E} f_i \cdot P_i$$

This penalizes variance in token distribution. $\alpha$ is a small coefficient (typically $10^{-2}$ to $10^{-3}$) so it doesn't dominate the language modeling loss. $f_i \cdot P_i$ is high when both actual routing frequency and router confidence are concentrated — penalizing both uneven routing and overconfident routing to dominant experts.

**Expert capacity buffer:** In practice, a hard capacity limit $C$ is set per expert per batch: $C = \frac{n_{\text{tokens}}}{E} \times \text{capacity\_factor}$. Tokens that would exceed capacity are dropped or handled by a fallback mechanism. This prevents one expert from being overwhelmed while ensuring the computation graph has a fixed shape (required for XLA/CUDA kernel efficiency).

### 10.4 MoE Training Instabilities

Beyond router collapse, MoE models suffer:

- **Expert load imbalance in distributed training:** If experts are sharded across devices (Expert Parallelism), uneven routing creates uneven device utilization — some GPUs idle while others are overloaded.
- **Fine-tuning instability:** MoE models fine-tune poorly with standard approaches because the router can re-collapse. Often requires freezing the router or using MoE-specific fine-tuning recipes.
- **Communication overhead:** In multi-device MoE (expert parallelism), tokens must be dynamically dispatched to the device holding their assigned expert, requiring all-to-all communication at every MoE layer.

---

## 11. Tokenization: The Invisible Bias Layer

### 11.1 The Design Space

Tokenization is a **data compression problem** with downstream consequences for model capability, fairness, and cost.

| Strategy | Vocab Size | Pros | Cons |
|---|---|---|---|
| Word-level | 100K+ | Intuitive | Vocabulary explosion, OOV problem |
| Character-level | ~100 | No OOV | Sequences too long, slow training |
| Subword (BPE) | 32K–128K | Balance | Language bias |
| Byte-level BPE | 256 base | Universal | Slightly longer sequences |

### 11.2 Byte Pair Encoding (BPE): Algorithm

**Sennrich et al., 2016.** Iterative merge algorithm:

1. Initialize vocabulary with all individual characters (or bytes)
2. Count all adjacent symbol pairs in the training corpus
3. Merge the most frequent pair into a new symbol
4. Repeat until vocabulary size $V$ is reached

**Example:**

```
Corpus: "low low lower lowest"
Initial: l o w _ , l o w e r _, l o w e s t _
Most frequent pair: (l, o) → merge to "lo"
Next: (lo, w) → "low"
Next: (low, _) → "low_"  
...
```

After training, "lower" might tokenize to `["low", "er"]`, while "unprecedented" tokenizes to `["un", "prec", "ed", "ent", "ed"]` — more fragments for rare words.

### 11.3 Byte-Level BPE

**GPT-2 innovation.** Initialize with the 256 raw UTF-8 byte values rather than characters. This guarantees:
- **Zero unknown tokens:** Any Unicode string (emojis, CJK characters, mathematical symbols) can be represented as a sequence of bytes, even if never seen during tokenizer training
- **Lossless representation:** The original string is always recoverable

The BPE merges then operate on bytes, learning common byte sequences. Common English subwords still compress efficiently; rare characters fall back to byte sequences.

### 11.4 Structural Bias: The Fairness and Performance Problem

BPE vocabularies are optimized for the **training corpus distribution**. If 90% of training data is English:

- Common English words: single tokens (`"the"`, `"language"`, `"model"`)
- French/German words: 1-2 tokens typically  
- Arabic/Hindi words: 3-5 tokens often
- Code: variable (Python keywords compress well; obscure APIs don't)

**Concrete consequence:** A sentence conveying the same semantic content requires more tokens in non-Western languages. This means:
1. **Context window consumed faster** for non-English users — less effective context per dollar
2. **Higher API cost** per semantic unit for non-English speakers (per-token pricing)
3. **Slower generation** in tokens-per-second terms (though semantic content-per-second is similar)
4. **Compression inequality:** The model has been exposed to far fewer gradient updates per concept for underrepresented languages

**Mitigation:** Modern tokenizers (Llama-3, Gemma) deliberately oversample non-English data during BPE training to improve multilingual token efficiency. Llama-3's tokenizer uses a 128K vocabulary (vs. Llama-2's 32K), significantly improving non-English and code token efficiency.

### 11.5 Tokenization Artifacts That Affect Model Behavior

- **The "leading space" problem:** `"Hello"` and `" Hello"` are different tokens. Prompts that start mid-sentence behave differently from those starting at natural boundaries.
- **Number tokenization:** `"2024"` might tokenize as `["20", "24"]` or `["2", "0", "2", "4"]` depending on the tokenizer. This is why arithmetic is harder than it appears — the model never "sees" the number as a unit.
- **Capitalization tokens:** `"NATO"` and `"nato"` are different tokens, potentially with different embedding vectors despite identical semantic content.
- **Whitespace sensitivity:** Python code with inconsistent indentation creates different token sequences, affecting code model performance.

---

## 12. The Decoder-Only Architecture: Why It Won

### 12.1 The Three Competing Paradigms (2018–2021)

**Encoder-only (BERT family):** Bidirectional attention — every token sees every other token simultaneously. Pre-trained with Masked Language Modeling (MLM): randomly mask 15% of tokens, predict them from context.

*Strength:* Deep bidirectional context → excellent for classification, NER, QA extraction.
*Weakness:* Cannot generate naturally. Generating token $t+1$ requires the full sequence including future tokens, which contradicts auto-regression.

**Encoder-Decoder (T5, BART):** Encoder reads input bidirectionally; decoder generates output auto-regressively, attending to encoder output via **cross-attention**.

*Strength:* Natural fit for seq2seq tasks (translation, summarization).
*Weakness:* Two networks to train and maintain. Cross-attention adds complexity and memory. The two components must be balanced in capacity.

**Decoder-only (GPT family):** Causal (left-to-right) attention only. Pre-trained with Causal Language Modeling (CLM): predict every next token given all previous tokens.

### 12.2 The Scaling Argument

**Why decoder-only scales better:**

1. **Architectural uniformity:** Every layer is identical — attention + FFN with causal mask. No cross-attention blocks, no separate encoder. This simplifies distributed training significantly: pipeline parallelism, tensor parallelism, and data parallelism all apply cleanly.

2. **Training efficiency:** CLM trains on every token as a prediction target, giving $n$ gradient signals per sequence of length $n$. MLM only trains on 15% of tokens. CLM is $\sim 6.7\times$ more sample-efficient in terms of gradient signal per token processed.

3. **Emergent capabilities:** As decoder-only models scaled past $\sim 10^{23}$ training FLOPs, they developed **in-context learning** (following examples in the prompt without weight updates), chain-of-thought reasoning, and instruction following — capabilities that emerged from the simple next-token prediction objective without explicit supervision for these tasks.

4. **Unified architecture for training and deployment:** The same forward pass is used during pre-training and inference. Encoder-decoder models require separate inference logic for the two components.

### 12.3 Why Causal Masking Forces World Model Learning

This is the deep argument for why CLM produces more capable models than MLM at scale.

With MLM (BERT): the model predicts `[MASK]` tokens from bidirectional context. The task is essentially **local pattern completion** — fill in the blank given surrounding words. The model doesn't need to model global coherence or causal structure.

With CLM (GPT): the model must predict the next word given **only past context**. To minimize prediction error across a diverse corpus (code, math, news, books, scientific papers), the model must internalize:
- Factual knowledge (what words typically follow in factual domains)
- Causal structure (if X causes Y, Y follows X in text describing the causal chain)
- Reasoning patterns (mathematical derivations proceed in order)
- Discourse structure (arguments have premises before conclusions)

The causal constraint forces the model to build an **implicit world model** rather than just memorizing local patterns. Instruction following emerges because "User: question\nAssistant:" is just a pattern where the statistically most likely continuation is the correct answer.

### 12.4 Residual Connections: The Gradient Highway

Every transformer block wraps its sublayers in a residual connection:

$$x_{l+1} = x_l + \text{Sublayer}(x_l)$$

**The gradient flow argument:** During backpropagation:

$$\frac{\partial \mathcal{L}}{\partial x_l} = \frac{\partial \mathcal{L}}{\partial x_{l+1}} \cdot \left(1 + \frac{\partial \text{Sublayer}(x_l)}{\partial x_l}\right)$$

The `+1` term creates a **direct gradient path** from the output layer to every earlier layer, bypassing the sublayer entirely. Even if $\frac{\partial \text{Sublayer}}{\partial x}$ is very small (vanishing), the gradient still flows back at full strength via the identity path.

**The residual stream perspective (Elhage et al., 2021):** Rather than thinking of each layer as transforming the representation, think of the model as maintaining a **residual stream** — a single vector per token that flows from input to output. Each attention head and FFN layer **reads from** this stream (via projection), computes some update, and **writes back to** it (via addition). The stream is never overwritten, only accumulated upon. This framing makes it clearer why residual networks scale — information is preserved and accumulated, not repeatedly transformed.

---

## 13. Training Dynamics: Optimization, Loss Landscapes, and Pathologies

### 13.1 The Loss Landscape Geometry

Training a neural network = navigating a loss function $\mathcal{L}(\theta)$ over a parameter space with hundreds of billions of dimensions. Standard intuitions from 2D/3D geometry break down completely.

**Local minima are rare in high dimensions.** A local minimum requires that every one of $|\theta|$ dimensions has a positive second derivative simultaneously. The probability of this by chance in a random function decreases exponentially with dimension. What dominates instead:

**Saddle points:** Points where the gradient is zero but the Hessian has both positive and negative eigenvalues — some directions curve up, others curve down. The gradient norm approaches zero, making the optimizer appear stuck. First-order optimizers (SGD, Adam) can stall here because gradients become arbitrarily small near the saddle.

**Sharp vs. flat minima:** The geometry of the minimum matters enormously for generalization. Define the sharpness as the maximum eigenvalue $\lambda_{\max}$ of the Hessian $H$ at the minimum:
- **Sharp minimum:** Large $\lambda_{\max}$, narrow basin. A small perturbation $\epsilon$ to $\theta$ increases loss by $\sim \lambda_{\max} \epsilon^2 / 2$. Sensitive to distribution shift.
- **Flat minimum:** Small $\lambda_{\max}$, wide basin. Robust to small perturbations, generalizes better.

**Poor conditioning:** The condition number $\kappa = \lambda_{\max} / \lambda_{\min}$ of the Hessian measures the aspect ratio of the loss landscape. If $\kappa \gg 1$, the loss surface is a steep narrow ravine. SGD with a fixed learning rate must use a small enough step to avoid bouncing off the steep walls, but this step is too small to make progress along the ravine floor. Adam directly addresses this.

### 13.2 Gradient Diagnostics: Identifying Training Failures

**Monitoring protocol in production training:**

```python
# Log these metrics every N steps
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
# ^ also applies clipping, returns pre-clip norm

for name, param in model.named_parameters():
    if param.grad is not None:
        writer.add_histogram(f'grad/{name}', param.grad, step)
        writer.add_scalar(f'grad_norm/{name}', param.grad.norm(), step)
```

**Vanishing gradient signature:**
- Early layer gradient norms → 0 while later layers have healthy norms
- Training loss plateaus despite non-zero overall gradient
- Symptom onset: often after many layers of sigmoid/tanh activations, or insufficient residual connections

**Exploding gradient signature:**
- Gradient norm spikes to 100s or 1000s
- Loss changes to `NaN` or `Inf`
- Parameters update by massive magnitudes in one step

**Plateau without vanishing/exploding:**
- Consistent non-zero gradient norms but loss doesn't decrease
- Suspects: learning rate too small, saddle point, loss landscape conditioning issue

### 13.3 The Vanishing Gradient Triad of Solutions

**1. Weight Initialization:**

He initialization (for ReLU activations):
$$W \sim \mathcal{N}\!\left(0, \sqrt{\frac{2}{n_{\text{in}}}}\right)$$

Xavier/Glorot initialization (for tanh/sigmoid):
$$W \sim \mathcal{N}\!\left(0, \sqrt{\frac{2}{n_{\text{in}} + n_{\text{out}}}}\right)$$

**Why this matters:** The variance of activations after a linear layer with $n_{\text{in}}$ inputs is $\text{Var}[\text{output}] = n_{\text{in}} \cdot \text{Var}[w] \cdot \text{Var}[x]$. If we set $\text{Var}[w] = 2/n_{\text{in}}$ (He), then $\text{Var}[\text{output}] = \text{Var}[x]$ — the variance is preserved layer to layer. This keeps the signal from exploding or vanishing purely due to initialization.

**2. Non-saturating activations (ReLU/SwiGLU):**

Sigmoid derivative: $\sigma'(x) = \sigma(x)(1 - \sigma(x)) \leq 0.25$. Maximum gradient contribution is 0.25 per layer. After 30 layers: $0.25^{30} \approx 10^{-18}$. Gradient is effectively zero.

ReLU derivative: $\max(0, 1)$ — either 1 or 0. For active neurons, gradient passes through completely unattenuated. No multiplicative shrinkage.

**3. Residual connections:** As discussed in 12.4. The identity path provides guaranteed gradient flow regardless of sublayer behavior.

### 13.4 Exploding Gradients: Gradient Clipping

**Global norm clipping:**
$$\text{if } \|\nabla\|_2 > \tau: \quad \nabla \leftarrow \tau \cdot \frac{\nabla}{\|\nabla\|_2}$$

This is **direction-preserving** — the gradient's direction (which parameters to update and in which direction) is unchanged. Only the step magnitude is capped.

**Why $\tau = 1.0$ is the common default:** Empirical finding across many large-scale training runs. Some models use 0.5 (more conservative) for training stability early on.

**Important nuance:** Gradient clipping is a symptom management tool, not a cure. If gradients routinely require clipping, the underlying cause (poor initialization, missing residual connections, too-high learning rate, numerical instability in specific operations) should be diagnosed and fixed.

### 13.5 Batch Size and the Generalization Paradox

**The conventional wisdom:** Larger batches give more accurate gradient estimates → better optimization → better model.

**The empirical reality (Keskar et al., 2016):** Large batch training converges to sharp minima; small batch training converges to flat minima, generalizing better.

**The mechanism:**

Define the noise scale of SGD as:
$$N \approx \frac{S}{B}$$

where $S$ is the variance of per-sample gradients and $B$ is batch size. This noise acts as implicit regularization — it prevents the optimizer from settling into small, sharp basins by continuously perturbing the trajectory.

**Large batches ($B$ → full dataset):** Gradient noise → 0. Optimizer takes deterministic steps. Follows the steepest descent path directly into the nearest minimum, which may be sharp and non-generalizing.

**Small batches:** High gradient noise. Optimizer takes noisy steps. Sharp minima are unstable under noise — the optimizer bounces out. Only wide flat minima are stable attractors for a noisy optimizer.

**The linear scaling rule (Goyal et al., Facebook AI, 2017):** When doubling batch size, multiply learning rate by 2. This approximately preserves the noise scale and allows large-batch training to reach similar generalization as small-batch training — but requires careful learning rate warm-up (increase linearly over the first ~5 epochs to avoid early instability).

**Practical implication:** Distributed training necessarily uses large effective batch sizes. Without the linear scaling rule + warm-up, you pay a generalization penalty proportional to the scale-up factor.

---

## 14. Optimizers: Adam vs. SGD in the Modern Context

### 14.1 Adam: Adaptive Moment Estimation

**Kingma & Ba, 2014.** Maintains per-parameter first and second moment estimates:

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t \quad \text{(first moment — running mean of gradient)}$$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2 \quad \text{(second moment — running mean of squared gradient)}$$

Bias correction (moments are initialized at 0, creating bias toward 0 early in training):
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

Parameter update:
$$\theta_t = \theta_{t-1} - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

**Intuition:** $\sqrt{\hat{v}_t}$ is the RMS of recent gradients for each parameter. Dividing by it normalizes the effective learning rate per parameter — parameters with historically large gradients get smaller updates, and vice versa. This directly solves the poor conditioning / ravine problem.

**The memory cost:**
For a 7B parameter model in BF16 (2 bytes each):
- Model weights: 14 GB
- Gradients: 14 GB  
- Adam $m_t$ (stored in FP32): 28 GB
- Adam $v_t$ (stored in FP32): 28 GB
- **Total: 84 GB** — on a single GPU with 80 GB HBM, this doesn't fit

This is why distributed training with optimizer state sharding (ZeRO Stage 1) is mandatory even for 7B models.

### 14.2 AdamW: Decoupled Weight Decay

The standard Adam with L2 regularization adds $\lambda \theta$ to the gradient before the moment update. But then the effective weight decay depends on the adaptive learning rate — parameters with large gradients (which have large $v_t$, hence small effective LR) receive proportionally less regularization. **Weight decay is not uniform.**

AdamW (Loshchilov & Hutter, 2017) decouples regularization from the gradient update:

$$\theta_t = \theta_{t-1} - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t - \eta \lambda \theta_{t-1}$$

The weight decay term $\eta \lambda \theta_{t-1}$ is applied directly, uniformly, without adaptive scaling. This is the correct implementation and is essentially universally used over vanilla Adam for LLM training.

### 14.3 SGD With Momentum for Final Quality

$$v_t = \mu v_{t-1} + g_t$$
$$\theta_t = \theta_{t-1} - \eta v_t$$

The momentum term $\mu$ (typically 0.9) accumulates gradient history, helping power through saddle points and noisy directions. Without momentum, SGD's noise is purely harmful; with momentum, it smooths the trajectory while retaining the flat-minimum-finding benefit of noise.

**Why SGD generalizes better than Adam:** Adam's aggressive per-parameter adaptation can route the optimizer into sharp minima that happen to be fast to reach — the adaptive learning rate eliminates the noise that would otherwise bounce it out. SGD with momentum, being less adaptive, retains more noise and finds flatter solutions.

**The practical tradeoff:**

| Dimension | Adam/AdamW | SGD + Momentum |
|---|---|---|
| Convergence speed | Fast (3-5× fewer steps) | Slow |
| Hyperparameter sensitivity | Low (robust defaults) | High (LR schedule critical) |
| Final generalization | Good | ~1-2% better |
| Memory overhead | $3\times$ model size | $1.5\times$ model size |
| Use case | Most production training | Flagship final runs |

### 14.4 Learning Rate Schedules

**Warm-up:** Start with a very small LR (e.g., $10^{-7}$) and increase linearly to the target LR over the first 1-5% of training steps. Without warm-up, the large initial random gradients combined with a full LR can cause catastrophic early updates that destroy the initialization.

**Cosine annealing:**
$$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\left(1 + \cos\!\left(\frac{\pi t}{T}\right)\right)$$

Smoothly decays LR to near-zero by the end of training. The slow annealing in the final phase allows the optimizer to settle into a minimum precisely rather than continuing to bounce.

**The Chinchilla finding (Hoffmann et al., 2022):** Learning rate decay should be matched to the total training token budget. Training longer than the LR schedule is tuned for leads to underperformance. This drove the shift toward compute-optimal training runs.

---

## End of Part 2

**Covered:** FFN as memory banks, SwiGLU, MoE architecture, router collapse + load balancing loss, BPE tokenization, byte-level BPE, tokenization bias, decoder-only architecture and the scaling argument for it, residual stream perspective, loss landscape geometry (saddle points, sharp/flat minima, conditioning), gradient diagnostics, vanishing/exploding gradient solutions, batch size and generalization paradox, Adam/AdamW/SGD tradeoffs, learning rate schedules.

**Coming in Part 3:**
- Pre-training objectives and data curation realities
- Fine-tuning: full fine-tuning, catastrophic forgetting, LoRA mechanics (full derivation), QLoRA with NF4 quantization
- RLHF: reward modeling, PPO, KL divergence penalty, alignment tax
- Precision formats: FP32, FP16, BF16 — numerical properties and why BF16 won
- ZeRO optimizer stages 1/2/3 — distributed training mechanics



# Part 3

# LLM & Transformer Architecture: Deep Technical Study Notes

## Part 3 — Pre-training, Fine-tuning, LoRA/QLoRA, RLHF, Precision Formats & Distributed Training

---

## 15. Pre-training: The Foundation

### 15.1 The Pre-training Objective

The canonical pre-training objective for decoder-only models is **Causal Language Modeling (CLM)**, also called next-token prediction. Given a sequence of tokens $x_1, x_2, \ldots, x_n$, the model is trained to maximize:

$$\mathcal{L}_{\text{CLM}} = -\sum_{t=1}^{n} \log P(x_t \mid x_1, \ldots, x_{t-1}; \theta)$$

This is the negative log-likelihood of the observed sequence. Minimizing it is equivalent to maximizing the probability the model assigns to the actual next token.

**Why this objective is remarkably powerful:** It is **self-supervised** — every token in every document is simultaneously a training input and a supervision signal. A 1 trillion token corpus yields exactly 1 trillion gradient signals, with zero human labeling cost. The diversity of the objective forces generalization: the same weights must predict the next word in Python code, a legal contract, a medical paper, and a Shakespearean sonnet. The only way to minimize loss across all these domains simultaneously is to build genuinely general representations.

**Perplexity as the training metric:**

$$\text{PPL} = \exp\!\left(\frac{1}{n}\sum_{t=1}^n -\log P(x_t \mid x_{<t})\right)$$

Perplexity is the exponentiated average negative log-likelihood per token. A perplexity of $k$ means the model is, on average, as uncertain as if it were choosing uniformly among $k$ equally likely options. State-of-the-art models achieve single-digit perplexity on held-out web text. **Crucially, perplexity alone does not predict downstream task performance** — this is the fundamental limitation of pre-training metrics.

---

### 15.2 Scaling Laws: The Chinchilla Result

**Kaplan et al. (2020)** established that model loss scales predictably with compute $C$, dataset size $D$, and parameters $N$:

$$\mathcal{L}(N, D) \approx \frac{A}{N^\alpha} + \frac{B}{D^\beta} + \mathcal{L}_\infty$$

where $\mathcal{L}_\infty$ is the irreducible loss (entropy of natural language). This suggested that for a fixed compute budget, model size should grow much faster than data size — leading to the era of large but undertrained models (GPT-3 at 300B tokens, 175B parameters).

**Hoffmann et al. (2022) — the Chinchilla correction:** A more careful empirical study found that **compute-optimal training requires equal scaling of parameters and tokens**: for every doubling of model size, you should also double the training data. The optimal ratio is approximately:

$$D_{\text{optimal}} \approx 20 \times N$$

Chinchilla (70B parameters, 1.4T tokens) outperformed Gopher (280B parameters, 300B tokens) on almost all benchmarks, using the same compute budget. This result fundamentally changed how the field approaches pre-training: Llama-1's 7B model was trained on 1T tokens (far beyond the Chinchilla optimum for inference efficiency), Llama-3's 8B model on 15T tokens.

**The inference-optimal vs. compute-optimal tension:** Chinchilla optimal is compute-optimal for a single training run. But for deployment, you want the **smallest model** that achieves a given quality level, because inference costs dominate at scale. Training a smaller model on more data (compute-suboptimal) produces a model that is cheaper to serve while achieving similar quality — this is the Llama philosophy.

---

### 15.3 Data Curation: The Unsung Determinant of Quality

The quality of pre-training data dominates model quality at matched compute. Common pipeline stages:

**1. Deduplication:** Near-duplicate documents cause the model to memorize rather than generalize. MinHash LSH is standard for fuzzy deduplication at web scale. Exact deduplication with suffix arrays (Lee et al., 2022) catches verbatim copies. Llama-3's data pipeline includes both exact and fuzzy dedup.

**2. Quality filtering:** Heuristic filters (perplexity on a reference model, character-to-token ratio, fraction of alphabetic characters) remove machine-generated spam and low-quality text. Classifier-based filtering (train a fastText classifier on curated vs. web data) is more powerful but computationally expensive.

**3. Harmful content removal:** Classifier-based removal of CSAM, violent content, and PII. URL blocklists for known harmful domains. This is both ethical and practical — training on harmful content degrades model alignment.

**4. Domain mixing:** The ratio of web text, books, code, scientific papers, and curated sources is a critical hyperparameter. Code data dramatically improves reasoning ability even on non-code tasks (hypothesis: code teaches the model sequential, causal, step-by-step reasoning). Llama-3's mix included significantly more code than Llama-2.

**5. Data ordering and curriculum:** Training on easier/shorter sequences first, then harder/longer ones can improve convergence, analogous to curriculum learning in supervised settings. Most large-scale training uses shuffled data (no curriculum) for simplicity, but some models experiment with structured curricula.

---

## 16. Fine-tuning: Adapting the Pre-trained Foundation

### 16.1 The Catastrophic Forgetting Problem

A pre-trained base model is a general-purpose next-token predictor. It is not an assistant. When you fine-tune aggressively on a narrow dataset (e.g., customer service transcripts), gradient descent optimizes the weights for the new task. Since gradient descent doesn't distinguish "useful general knowledge" from "inefficient parameters to update," it can overwrite general capabilities.

**Formally:** Let $\theta^*$ be the pre-trained weights minimizing $\mathcal{L}_{\text{pre-train}}(\theta)$ and $\theta'$ be the fine-tuned weights minimizing $\mathcal{L}_{\text{fine-tune}}(\theta)$. If the fine-tuning distribution is narrow and the learning rate is high, $\theta'$ can be far from $\theta^*$ in weight space, with $\mathcal{L}_{\text{pre-train}}(\theta') \gg \mathcal{L}_{\text{pre-train}}(\theta^*)$. The model has "forgotten" its pre-training.

**Mitigations in full fine-tuning:**
- **Low learning rate:** $\eta \sim 10^{-5}$ to $10^{-6}$ vs. $10^{-3}$ to $10^{-4}$ for pre-training
- **Distribution mixing:** Keep 10-30% pre-training data in the fine-tuning mix to anchor weights
- **Elastic Weight Consolidation (EWC):** Add a regularization term $\lambda \sum_i F_i (\theta_i - \theta_i^*)^2$ where $F_i$ is the Fisher information — the importance of each parameter to the pre-training objective. Penalizes moving important parameters. Computationally expensive at LLM scale.
- **LoRA:** The practical solution — freeze the base model entirely, train only small adapter matrices

---

### 16.2 LoRA: Low-Rank Adaptation — Full Derivation

**Hu et al., 2021.** The central hypothesis: **the weight updates required to adapt a pre-trained model to a new task have a low intrinsic dimensionality** — they lie in a low-dimensional subspace of the full parameter space.

**Mathematical basis for this hypothesis:** Aghajanyan et al. (2020) showed that fine-tuning objectives can be minimized by optimizing in subspaces of dimension as low as 100-1000, regardless of the full parameter count. The pre-trained model already spans the relevant representation space; fine-tuning is a small directional adjustment within it.

**The LoRA construction:**

For a weight matrix $W_0 \in \mathbb{R}^{m \times n}$ (frozen), instead of learning a full $\Delta W \in \mathbb{R}^{m \times n}$, we constrain the update to rank $r$:

$$\Delta W = BA$$

where $B \in \mathbb{R}^{m \times r}$, $A \in \mathbb{R}^{r \times n}$, with $r \ll \min(m, n)$.

The modified forward pass:

$$h = W_0 x + \Delta W x = W_0 x + BA x$$

**Initialization:**
- $A$ is initialized with random Gaussian values: $A \sim \mathcal{N}(0, \sigma^2)$
- $B$ is initialized to **zero**: $B = 0$

This guarantees $\Delta W = BA = 0$ at the start of training, meaning LoRA begins exactly at the pre-trained weights. Training starts from a stable baseline.

**The scaling factor $\alpha$:**

In practice, the update is scaled:
$$h = W_0 x + \frac{\alpha}{r} BA x$$

$\alpha$ is a fixed hyperparameter (not trained). Setting $\alpha = r$ gives $\alpha/r = 1$ — equivalent to no scaling. The scaling factor allows tuning the magnitude of the LoRA contribution independently of $r$. Common practice: set $\alpha = 2r$ or $\alpha = 16$ while varying $r$.

**Parameter reduction analysis:**

For a single attention projection $W^Q \in \mathbb{R}^{4096 \times 4096}$:
- Full fine-tuning trainable parameters: $4096^2 = 16,777,216$
- LoRA with $r = 8$: $4096 \times 8 + 8 \times 4096 = 65,536$
- **Reduction factor: 256×**

For a full model with LoRA applied to all attention projections ($W^Q, W^K, W^V, W^O$ in each of $L = 32$ layers):
$$\text{LoRA params} = 4 \times 2 \times d_{model} \times r \times L = 4 \times 2 \times 4096 \times 8 \times 32 = 8,388,608$$

A 7B parameter model fine-tuned with LoRA trains only ~8M parameters — **0.12% of total parameters**.

**Selecting $r$:**

- $r = 4$: Minimal adaptation, very fast, suitable for style/format changes
- $r = 8$: The practical default for most instruction fine-tuning
- $r = 16$ to $r = 64$: For complex domain adaptation (medical, legal, code)
- $r = 256$+: Approaches full fine-tuning expressivity; rarely justified

**Which matrices to apply LoRA to:**

Original paper: $W^Q$ and $W^V$ only. Subsequent work found applying to all four attention projections ($W^Q, W^K, W^V, W^O$) and even FFN layers ($W_1, W_2$) consistently improves quality. The tradeoff is more trainable parameters.

**Merging at inference:**

After training, the LoRA matrices can be merged back into the base weights:
$$W_{\text{merged}} = W_0 + \frac{\alpha}{r} BA$$

This adds **zero inference overhead** — the merged model is identical in architecture to the base model. No adapter layers, no additional FLOPs. This is a major practical advantage over other PEFT approaches that require permanent adapter layers at inference time.

---

### 16.3 QLoRA: Quantized LoRA — Democratizing Fine-tuning

**Dettmers et al., 2023.** Combines three techniques to enable fine-tuning of 65B+ parameter models on a single consumer GPU:

**Technique 1: 4-bit NormalFloat (NF4) Quantization**

Standard quantization maps a range of values to $2^k$ discrete levels. For 4-bit: 16 discrete levels. The choice of where to place those 16 levels matters enormously.

NF4 uses a **quantile-based placement**: the 16 quantile boundaries of a standard normal distribution $\mathcal{N}(0, 1)$. Pre-trained model weights are empirically approximately normally distributed (a consequence of random initialization and gradient descent). NF4 places quantization levels where the data actually is, minimizing quantization error.

$$q_i = \frac{1}{2}\left(Q_X^{-1}\!\left(\frac{i}{2^k + 1}\right) + Q_X^{-1}\!\left(\frac{i+1}{2^k + 1}\right)\right)$$

where $Q_X^{-1}$ is the quantile function of the empirical weight distribution.

**Comparison to standard int4:** Standard uniform int4 places levels at equal intervals, wasting quantization capacity in the tails where few weights exist. NF4 concentrates levels where weights are dense (near zero), significantly reducing quantization error for normally-distributed weights.

**Technique 2: Double Quantization**

Quantization requires storing quantization constants (scale factors and zero points) for each block of weights. These constants themselves consume memory. QLoRA quantizes the quantization constants — "quantizing the quantizer."

For a blocksize of 64 weights with FP32 quantization constants: each constant takes 32 bits for 64 weights = 0.5 bits/parameter overhead. After double quantization (to 8-bit constants with blocksize 256): 8/256 = 0.03 bits/parameter overhead. **Net reduction: 0.47 bits/parameter** — small but meaningful at scale.

**Technique 3: Paged Optimizers**

NVIDIA unified memory allows CPU RAM to act as overflow storage for GPU memory. During training, optimizer states (the large Adam moment matrices) are stored in CPU RAM when GPU memory is tight and paged into GPU memory only during the weight update step. This allows brief memory spikes during backward pass without OOM crashes.

**Hardware requirement comparison (7B model):**

| Method | Precision | VRAM Required |
|---|---|---|
| Full fine-tuning | FP16 | ~112 GB (model + grad + optimizer) |
| LoRA | FP16 | ~28 GB (frozen model + grad + small optimizer state) |
| QLoRA | NF4 + FP16 adapters | ~6-8 GB |

QLoRA puts 7B model fine-tuning on an RTX 3090 (24 GB VRAM). 13B models on an A6000 (48 GB). **This is the democratization event for LLM development.**

**The quality cost of QLoRA:** Dettmers et al. showed QLoRA fine-tuned models match full 16-bit fine-tuning performance on most benchmarks. The NF4 quantization loss is compensated by LoRA's targeted adaptation. For most practical fine-tuning tasks, the quality difference is negligible.

---

## 17. RLHF: Alignment From Human Feedback

### 17.1 Why CLM-trained Models Need Alignment

A base model trained on CLM is a **document continuation engine**. Given "How do I make a bomb?", the statistically likely continuation in training data might be an actual explanation (from chemistry textbooks, fictional novels, security research papers). The model has no concept of "this is harmful to output."

More subtly, instruction-following emerges weakly from CLM. The model can follow instructions in-context (because instructions followed by responses are common in training data), but it's unreliable and inconsistent.

**The alignment goal:** Make the model reliably helpful, harmless, and honest (HHH) — not by restricting its knowledge, but by changing its behavioral policy to produce outputs that humans prefer.

### 17.2 The Three-Stage RLHF Pipeline

**Stage 1: Supervised Fine-tuning (SFT)**

Collect high-quality demonstration data: human-written responses to diverse prompts. Fine-tune the base model on this data with standard CLM loss. This creates the **SFT model** — a well-behaved starting point for RL training.

SFT alone produces significant improvement in instruction following. InstructGPT (Ouyang et al., 2022) showed that 1.3B parameter SFT models outperform 175B base models on human preference evaluations — alignment quality matters more than raw scale for user-facing tasks.

**Stage 2: Reward Model Training**

Collect **comparison data**: human raters evaluate multiple model responses to the same prompt and rank them by preference. The reward model $r_\phi(x, y)$ is trained to predict human preference from (prompt, response) pairs.

Training objective: For a prompt $x$ with preferred response $y_w$ and rejected response $y_l$:

$$\mathcal{L}_{\text{RM}} = -\mathbb{E}_{(x, y_w, y_l) \sim D}\!\left[\log \sigma\!\left(r_\phi(x, y_w) - r_\phi(x, y_l)\right)\right]$$

This is the **Bradley-Terry preference model** — it trains the reward model to assign higher scores to preferred responses. $\sigma$ is the sigmoid function; the loss approaches 0 when $r_\phi(x, y_w) \gg r_\phi(x, y_l)$.

**Reward model architecture:** Typically the SFT model with the final token prediction head replaced by a scalar output head. The scalar is the reward for the entire response.

**The reward hacking problem (Goodhart's Law):** The reward model is an imperfect proxy for human preference. As RL optimization pressure increases, the policy learns to exploit reward model weaknesses — producing responses that score high on the proxy but are not actually preferred by humans. Classic examples: extremely verbose responses (length bias in annotators), excessive hedging, sycophantic agreement with stated user preferences.

**Stage 3: RL Fine-tuning with PPO**

Use the reward model as the reward signal and optimize the SFT model (policy) using **Proximal Policy Optimization (PPO)**:

$$\mathcal{L}_{\text{PPO}} = \mathbb{E}_{(x,y) \sim \pi_\theta}\!\left[r_\phi(x, y) - \beta \cdot D_{\text{KL}}\!\left[\pi_\theta(\cdot \mid x) \;\|\; \pi_{\text{ref}}(\cdot \mid x)\right]\right]$$

**The KL divergence penalty** is critical:

$$D_{\text{KL}}[\pi_\theta \| \pi_{\text{ref}}] = \sum_y \pi_\theta(y \mid x) \log \frac{\pi_\theta(y \mid x)}{\pi_{\text{ref}}(y \mid x)}$$

where $\pi_{\text{ref}}$ is the frozen SFT model. This term penalizes the policy for drifting too far from the SFT model.

**Why the KL penalty is essential:**
- Without it, PPO can collapse the policy to generate only the handful of response patterns that happen to score high on the reward model (mode collapse)
- The KL penalty keeps the policy in the neighborhood of the SFT model, preserving diversity and pre-trained knowledge
- It prevents reward hacking from escalating to complete gibberish

**The $\beta$ tradeoff:** Large $\beta$ → policy stays close to SFT, minimal reward maximization. Small $\beta$ → aggressive reward maximization, risk of reward hacking and capability degradation. Tuning $\beta$ is one of the critical RLHF hyperparameters. Typical values: $0.1$ to $0.5$.

### 17.3 The Alignment Tax

Empirical finding across many models: RLHF alignment consistently reduces performance on certain capability benchmarks — particularly open-ended generation, creative tasks, and tasks requiring the model to reason through unconventional solution paths.

**The mechanism:** PPO with KL penalty effectively shrinks the model's output distribution toward a safe, conventional mode. The model learns to prefer high-reward, safe, predictable responses. Low-probability but potentially brilliant or creative token sequences get systematically suppressed because they're risky — they might score poorly on the reward model.

**The intellectual timidity problem:** Aligned models tend to hedge, qualify, and default to "I'm not sure" more than base models. They avoid making confident claims even when the base model would be correct. This is the reward model's length and hedging bias being exploited.

**DPO as an alternative (Rafailov et al., 2023):** Direct Preference Optimization bypasses the separate reward model entirely. It shows that the optimal RLHF policy has a closed-form solution that can be expressed as a classification objective directly on preference pairs:

$$\mathcal{L}_{\text{DPO}} = -\mathbb{E}\!\left[\log \sigma\!\left(\beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}\right)\right]$$

DPO is simpler to implement (no RL, no reward model), more stable to train, and produces comparable or better results. Most modern alignment pipelines use DPO or variants (IPO, KTO) rather than PPO.

---

## 18. Numerical Precision: FP32, FP16, BF16, FP8

### 18.1 IEEE 754 Floating Point Formats

A floating point number is represented as:

$$(-1)^s \times 2^{e - \text{bias}} \times (1 + \text{mantissa})$$

| Format | Total bits | Sign | Exponent | Mantissa | Max value | Min positive |
|---|---|---|---|---|---|---|
| FP32 | 32 | 1 | 8 | 23 | $\sim 3.4 \times 10^{38}$ | $\sim 1.2 \times 10^{-38}$ |
| FP16 | 16 | 1 | 5 | 10 | $65,504$ | $\sim 6.1 \times 10^{-5}$ |
| BF16 | 16 | 1 | 8 | 7 | $\sim 3.4 \times 10^{38}$ | $\sim 1.2 \times 10^{-38}$ |
| FP8 (E4M3) | 8 | 1 | 4 | 3 | $448$ | $\sim 0.002$ |

### 18.2 Why FP16 Fails for Deep Learning

FP16's 5-bit exponent gives a maximum representable value of $65,504$. In deep neural networks:

**Overflow:** Intermediate activations and gradients during backward pass can easily exceed 65,504, especially in deep networks with large batch sizes. When overflow occurs, the value becomes `Inf`. Once `Inf` enters the computation graph, it propagates: `Inf - Inf = NaN`. The training run is dead.

**Underflow:** Very small gradients (which occur naturally in deep networks) can fall below the FP16 minimum positive value of ~$6 \times 10^{-5}$, rounding to exactly zero. The gradient signal vanishes at that parameter permanently.

**The loss scaling hack:** To use FP16, practitioners multiplied the loss by a large scalar (e.g., 1024 or 32768) before backward, scaling all gradients up into the FP16 representable range. Then divide by the same scalar before the weight update. This required:
- Monitoring for overflow (NaN/Inf) and reducing the scale factor when it occurs
- Re-running the step if overflow was detected
- Complex infrastructure to manage the dynamic scaling factor

Fragile, slow, and required careful tuning. Any overflow detection failure corrupted the training run.

### 18.3 BF16: The Elegant Fix

**Brain Float 16 (Google Brain, 2018).** The key insight: for neural network training, **dynamic range matters more than precision**. The model needs to represent very large numbers (forward pass activations) and very small numbers (gradients) without overflow or underflow. It doesn't need 10 bits of mantissa precision — 7 bits is sufficient for stochastic gradient descent to converge.

BF16 allocates 8 bits to the exponent (same as FP32) and only 7 bits to the mantissa. This gives **FP32 dynamic range in 16 bits**, completely eliminating overflow and underflow as practical concerns. Loss scaling becomes unnecessary.

**The precision tradeoff:** BF16 has $2^7 = 128$ discrete levels between powers of 2, vs. FP32's $2^{23} = 8,388,608$. The numbers are "blurrier." But SGD-based optimizers don't need high precision — they're averaging over many stochastic samples anyway. The direction of the gradient matters more than its precise magnitude. Empirically, BF16 training matches FP32 quality while using half the memory and achieving higher throughput (hardware can perform more BF16 ops/second than FP32).

**Mixed precision training (standard pattern):**
- Model weights: BF16 (forward pass, backward pass)
- Optimizer states (Adam moments): FP32 (weight updates require precision to accumulate small incremental changes over many steps)
- Gradients: BF16 during backward, converted to FP32 for optimizer step

This is the default in PyTorch's `torch.cuda.amp` with `dtype=torch.bfloat16`.

### 18.4 FP8: The Frontier (H100 and Beyond)

NVIDIA H100 introduced hardware FP8 support (E4M3 and E5M2 formats). FP8 halves the memory of BF16 and doubles throughput for matrix multiplications. Used for **inference quantization** and experimentally for training (Transformer Engine in Hopper architecture).

FP8 training requires careful per-tensor scaling to prevent the extremely limited range (max 448 in E4M3) from causing overflow in activations. NVIDIA's Transformer Engine handles this automatically using calibrated scaling factors per operation.

---

## 19. Distributed Training: ZeRO and Model Parallelism

### 19.1 The Memory Wall

For a 70B parameter model in BF16:
- Model weights: $70 \times 10^9 \times 2 \text{ bytes} = 140 \text{ GB}$
- Gradients (same size): $140 \text{ GB}$
- Adam optimizer states (FP32): $70 \times 10^9 \times 2 \times 4 \text{ bytes} = 560 \text{ GB}$
- **Total: ~840 GB**

An H100 has 80 GB HBM. A single GPU cannot hold even the weights, let alone the full training state. Distributed training is mandatory.

### 19.2 Data Parallel Training (Baseline)

Each GPU holds a **complete copy** of the model. The dataset is partitioned across GPUs. Each GPU computes gradients on its local data shard, then gradients are **all-reduced** (summed and averaged) across all GPUs using collective communication (NCCL's ring all-reduce).

**Memory issue:** Every GPU holds the full model + full gradients + full optimizer state. For 70B: 840 GB per GPU. Impossible.

**Compute issue:** Communication of gradients ($140 \text{ GB}$) must happen every step. On 8 GPUs over NVLink (600 GB/s): $140 / 600 \approx 0.23$ seconds per step just for communication. This becomes a bottleneck at scale.

### 19.3 ZeRO: Zero Redundancy Optimizer

**Rajbhandari et al. (DeepSpeed, 2020).** ZeRO eliminates the redundant copies of model state across data-parallel processes. Three stages of increasing aggressiveness:

**ZeRO Stage 1 — Optimizer State Partitioning:**

Instead of every GPU holding all Adam states, GPU $k$ holds only the optimizer states for $\frac{1}{N}$ of the parameters (its assigned partition). During the optimizer step, each GPU updates only its shard. A subsequent all-gather reconstructs the full updated weights.

$$\text{Memory per GPU} = \frac{1}{N}\left(12 \text{ bytes/param}\right) + 4 \text{ bytes/param (weights)} + 2 \text{ bytes/param (gradients)}$$

For $N=64$ GPUs: optimizer state per GPU reduces from 560 GB to 8.75 GB. **Total per GPU: ~18 GB for 70B model.**

**ZeRO Stage 2 — Gradient Partitioning:**

Additionally, each GPU only stores gradients for its parameter partition. Gradients are reduced directly into the responsible GPU's shard rather than all-reducing the full gradient tensor.

$$\text{Memory per GPU} \approx \frac{14 \times \text{bytes/param}}{N} + 2 \text{ bytes/param (weights)}$$

For $N=64$: each GPU needs approximately **2.2 GB for optimizer+gradients + 140 GB weights** ... but the weights themselves still need to be replicated for the forward/backward pass. Stage 2 alone gets you:

**ZeRO Stage 3 — Parameter Partitioning:**

The most radical stage. Each GPU holds only $\frac{1}{N}$ of the model weights. No GPU has the full model at any time.

$$\text{Memory per GPU} = \frac{16 \text{ bytes/param}}{N}$$

For $N=64$: **$\approx 840 / 64 \approx 13.1 \text{ GB per GPU}$**. A 70B model fits on 64 GPUs with a single A100 (40 GB) per GPU — feasible.

**The communication cost of Stage 3:**

During the forward pass, as computation proceeds layer by layer, each GPU must **all-gather** the parameters for the current layer from all other GPUs. After computing that layer, the gathered parameters are immediately **discarded** from local memory (to free space for the next layer). During backward, the same all-gather + discard pattern occurs, followed by a **reduce-scatter** of gradients into the owning GPU's partition.

This doubles the communication volume compared to Stage 1/2, but the communication can be **overlapped with computation** (pre-fetching the next layer's parameters while computing the current layer). With NVLink (600 GB/s) or InfiniBand (200 Gb/s), this overlap makes Stage 3 practical.

**ZeRO-Infinity:** Extends Stage 3 to offload parameters and optimizer states to CPU RAM and even NVMe SSDs when GPU memory is exhausted. Enables trillion-parameter model training on GPU clusters with aggressive CPU offloading.

### 19.4 Tensor Parallelism (Megatron-LM)

ZeRO is a data parallelism optimization. **Tensor Parallelism** partitions individual weight matrices across devices.

For an FFN layer $Y = \text{GeLU}(XA)B$ with $A \in \mathbb{R}^{d \times 4d}$, $B \in \mathbb{R}^{4d \times d}$:

**Column-parallel (for A):** Partition $A$ by columns: $A = [A_1, A_2]$ where each half is on a different GPU. Each GPU computes $Y_i = \text{GeLU}(XA_i)$ independently (no communication). **Row-parallel (for B):** Partition $B$ by rows: $B = [B_1; B_2]$ where $Y = Y_1 B_1 + Y_2 B_2$. Each GPU computes its contribution; a single **all-reduce** sums the results.

Result: one all-reduce per FFN layer (instead of per-parameter communication). Much more communication-efficient than ZeRO Stage 3 for this case.

**Attention is tensor-parallelized** by partitioning attention heads across GPUs — each GPU owns a subset of heads. No communication needed within the attention computation; one all-reduce after the output projection.

### 19.5 Pipeline Parallelism

For models too large even for tensor parallelism across available devices, **pipeline parallelism** partitions the model by layer depth: GPU 1 holds layers 1-10, GPU 2 holds layers 11-20, etc. The forward pass "pipelines" microbatches through the stages.

**The bubble problem:** Naive pipeline parallelism (GPipe) has a pipeline bubble — GPU 1 is idle during the backward pass while GPU 4 finishes its forward pass. Bubble fraction: $(p-1)/(m+p-1)$ where $p$ is pipeline depth and $m$ is number of microbatches. With $p=8$ stages and $m=32$ microbatches: $(8-1)/(32+8-1) = 7/39 \approx 18\%$ idle time.

**Interleaved scheduling (Megatron-LM v2)** assigns non-contiguous layers to each GPU (e.g., GPU 1 has layers 1-5 and 41-45) using virtual stages, reducing the bubble fraction at the cost of more communication.

**3D Parallelism:** Production LLM training (e.g., Megatron-Turing NLG 530B, GPT-4) combines all three: Data Parallelism (ZeRO) + Tensor Parallelism + Pipeline Parallelism. The 3D combination allows scaling to thousands of GPUs while maintaining efficiency.

---

## 20. PyTorch Training Hygiene: Silent Failure Modes

These are issues that experienced engineers still encounter and that can waste days of debugging.

**`model.train()` vs `model.eval()`:**
- `model.train()`: Dropout is active (randomly zeroing activations for regularization), BatchNorm uses batch statistics
- `model.eval()`: Dropout is disabled (full network used), BatchNorm uses running statistics
- **Failure mode:** Evaluating with `model.train()` gives noisy, unreproducible metrics. Training with `model.eval()` disables dropout regularization, leading to overfitting without warning. Always set mode explicitly.

**`optimizer.zero_grad()`:**
- PyTorch **accumulates** gradients by default. If you don't zero gradients before each backward pass, you're adding gradients from multiple batches together
- **Failure mode:** Loss appears to decrease slowly or erratically. Gradient norms grow unexpectedly. The bug is silent — no error, just wrong behavior
- **Efficient pattern:** `optimizer.zero_grad(set_to_none=True)` sets gradients to `None` rather than zero tensors, saving memory allocation

**`torch.no_grad()` during inference:**
- Without this context manager, PyTorch builds the computation graph (autograd tape) even during inference, consuming memory proportional to the forward pass
- **Failure mode:** OOM during inference that shouldn't occur given the model size

**Gradient accumulation:**
For large effective batch sizes that don't fit in GPU memory, accumulate gradients over $k$ forward-backward passes before updating weights:
```python
for i, batch in enumerate(dataloader):
    loss = model(batch) / accumulation_steps
    loss.backward()  # accumulates into .grad
    if (i + 1) % accumulation_steps == 0:
        optimizer.step()
        optimizer.zero_grad()
```
**Failure mode:** Forgetting to divide loss by `accumulation_steps` scales the effective learning rate by $k$, causing training instability.

**Random seed management in distributed training:**
Each process must have the same model initialization seed but different data sampling seeds. Failure to set different data seeds leads to all GPUs training on identical data — effectively a $1/N$ reduction in effective dataset size.

---

## End of Part 3

**Covered:** Pre-training objectives and scaling laws (Chinchilla), data curation pipeline, catastrophic forgetting, full fine-tuning mitigations, LoRA (complete mathematical derivation, rank selection, merging), QLoRA (NF4, double quantization, paged optimizers, hardware requirements), RLHF (SFT, reward modeling, PPO with KL penalty, alignment tax), DPO, floating point formats (FP16 failure modes, BF16 design, FP8), ZeRO Stages 1/2/3, tensor parallelism, pipeline parallelism, 3D parallelism, PyTorch training pitfalls.

**Coming in Part 4 (Final):**
- RAG: naive vs. advanced architectures, chunking theory, re-ranking with cross-encoders
- Vector databases: HNSW vs. IVF — algorithmic deep dive
- Inference optimization: prefill vs. decode, KV cache fragmentation, PagedAttention
- Speculative decoding: mathematical guarantee of correctness
- The "lost in the middle" problem and long-context architecture futures



# Part 4